<a href="https://colab.research.google.com/github/saha23s/CD-MSC/blob/aaron%2Fpreprocessing/colab_quickstart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CD-MSC Baseline — Colab Quickstart
Run cells top to bottom. Make sure you are on a GPU runtime before starting:
**Runtime → Change runtime type → T4 GPU**

In [1]:
# Confirm GPU is available
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU found — check Runtime settings')

True
Tesla T4


In [2]:
# Clone the repo and switch to your branch
!git clone https://github.com/saha23s/CD-MSC.git
%cd CD-MSC
!git checkout aaron/preprocessing

fatal: destination path 'CD-MSC' already exists and is not an empty directory.
/content/CD-MSC
Already on 'aaron/preprocessing'
Your branch is up to date with 'origin/aaron/preprocessing'.


In [3]:
# Install dependencies
!pip install -r requirements.txt

In [4]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Check what the zip is called and where it is
!ls "/content/drive/MyDrive/"

In [6]:
# Extract the raw audio into the right place
# Update the zip filename below to match what you see in the cell above
ZIP_PATH = "/content/drive/MyDrive/Development_data.zip"  # <-- update if needed

!unzip -q "{ZIP_PATH}" -d Development_data/

# Confirm the expected structure is in place
!ls Development_data/

Development_data  __MACOSX


In [8]:
!find Development_data/ -name "*.wav" | head -5

Development_data/Development_data/raw_audio/S_3_D_5_7753.wav
Development_data/Development_data/raw_audio/S_3_D_5_74261.wav
Development_data/Development_data/raw_audio/S_4_D_5_42112.wav
Development_data/Development_data/raw_audio/S_1_D_5_78120.wav
Development_data/Development_data/raw_audio/S_2_D_5_5547.wav


In [9]:
!mv Development_data/Development_data/* Development_data/
!rm -rf Development_data/Development_data/


In [10]:
# Confirm raw audio is accessible
!ls Development_data/raw_audio/ | head -5
!ls Development_data/raw_audio/ | wc -l

S_1_D_1_100.wav
S_1_D_1_101.wav
S_1_D_1_102.wav
S_1_D_1_103.wav
S_1_D_1_104.wav
271380


In [11]:
# Extract log-mel features for all splits
# This reads raw_audio/ and writes .pkl files to Development_data/feature/
# Skips automatically if feature files already match the config
!python extract_features.py --config configs/default_experiment.json

Streaming output truncated to the last 5000 lines.
[test] 22219/27217 | id=S_5_D_1_127 | feature_shape=(257, 64)
[test] 22220/27217 | id=S_5_D_1_1270 | feature_shape=(1793, 64)
[test] 22221/27217 | id=S_5_D_1_1271 | feature_shape=(257, 64)
[test] 22222/27217 | id=S_5_D_1_1272 | feature_shape=(769, 64)
[test] 22223/27217 | id=S_5_D_1_1273 | feature_shape=(513, 64)
[test] 22224/27217 | id=S_5_D_1_1274 | feature_shape=(3323, 64)
[test] 22225/27217 | id=S_5_D_1_1275 | feature_shape=(2305, 64)
[test] 22226/27217 | id=S_5_D_1_1276 | feature_shape=(1537, 64)
[test] 22227/27217 | id=S_5_D_1_1277 | feature_shape=(513, 64)
[test] 22228/27217 | id=S_5_D_1_1278 | feature_shape=(257, 64)
[test] 22229/27217 | id=S_5_D_1_1279 | feature_shape=(1275, 64)
[test] 22230/27217 | id=S_5_D_1_128 | feature_shape=(513, 64)
[test] 22231/27217 | id=S_5_D_1_1280 | feature_shape=(257, 64)
[test] 22232/27217 | id=S_5_D_1_1281 | feature_shape=(3329, 64)
[test] 22233/27217 | id=S_5_D_1_1282 | feature_shape=(257, 64)


In [12]:
import shutil
shutil.copytree('Development_data/feature', '/content/drive/MyDrive/CD-MSC-feature', dirs_exist_ok=True)
print('Feature files saved to Drive.')


Feature files saved to Drive.


In [ ]:
import shutil
shutil.copytree('/content/drive/MyDrive/CD-MSC-feature', 'Development_data/feature', dirs_exist_ok=True)
print('Feature files restored.')


In [13]:
# Train — runs for up to 100 epochs with early stopping
# Expects ~15-20 epochs before stopping; should take 20-40 min on GPU
!python train.py --config configs/default_experiment.json

training model to outputs/MTRCNN_seed42_B64_E100_earlystop_min10_pati5
loading from Development_data/feature/training_features.pkl
loading from Development_data/feature/validation_features.pkl
loading from Development_data/feature/training_feature_stats.json
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware th

In [14]:
# Save outputs to Drive before the session expires
import shutil
shutil.copytree('outputs', '/content/drive/MyDrive/CD-MSC-outputs', dirs_exist_ok=True)
print('Outputs saved to Google Drive.')

Outputs saved to Google Drive.


In [15]:
!python evaluate.py --config configs/default_experiment.json --checkpoint outputs/MTRCNN_seed42_B64_E100_earlystop_min10_pati5/model/model_best.pth --split test --metrics-out outputs/MTRCNN_seed42_B64_E100_earlystop_min10_pati5/manual_test_metrics.json --predictions-out outputs/MTRCNN_seed42_B64_E100_earlystop_min10_pati5/manual_test_predictions.jsonl


loading from outputs/MTRCNN_seed42_B64_E100_earlystop_min10_pati5/model/model_best.pth
loading from Development_data/feature/test_features.pkl
loading from Development_data/feature/training_feature_stats.json
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
{
  "loss": 1.8774332988368942,
  "species_loss": 1.456659212966865,
  "domain_loss": 0.4207740881919289,
  "species_accuracy": 0.7825990915298462,
  "species_balanced_accuracy": 0.5422473723689715,
  "domain_accuracy": 0.9215931296348572,
  "domain_balanced_accuracy": 0.45997379124164584,
  "species_ba_D1": 0.414930563129019,
  "spec

In [16]:
!python predict.py --config configs/default_experiment.json --checkpoint outputs/MTRCNN_seed42_B64_E100_earlystop_min10_pati5/model/model_best.pth --audio Development_data/raw_audio/S_1_D_5_16608.wav


loading from outputs/MTRCNN_seed42_B64_E100_earlystop_min10_pati5/model/model_best.pth
loading from Development_data/feature/training_feature_stats.json
{
  "predicted_species": "Aedes aegypti",
  "probabilities": {
    "Aedes aegypti": 0.997615,
    "Aedes albopictus": 8.6e-05,
    "Culex quinquefasciatus": 3e-06,
    "Anopheles gambiae": 0.001107,
    "Anopheles arabiensis": 2.8e-05,
    "Anopheles dirus": 0.0,
    "Culex pipiens": 0.001161,
    "Anopheles minimus": 0.0,
    "Anopheles stephensi": 0.0
  }
}


In [17]:
!python run_multi_seed_experiments.py --config configs/multi_seed_experiment.json


loading from outputs/MTRCNN_seed3407_B64_E100_earlystop_min10_pati5/run_summary.json
loading from outputs/MTRCNN_seed1234_B64_E100_earlystop_min10_pati5/run_summary.json
loading from outputs/MTRCNN_seed2023_B64_E100_earlystop_min10_pati5/run_summary.json
loading from outputs/MTRCNN_seed2024_B64_E100_earlystop_min10_pati5/run_summary.json
loading from outputs/MTRCNN_seed1024_B64_E100_earlystop_min10_pati5/run_summary.json
loading from outputs/MTRCNN_seed2048_B64_E100_earlystop_min10_pati5/run_summary.json
loading from outputs/MTRCNN_seed4096_B64_E100_earlystop_min10_pati5/run_summary.json
loading from outputs/MTRCNN_seed8192_B64_E100_earlystop_min10_pati5/run_summary.json
loading from outputs/MTRCNN_seed10086_B64_E100_earlystop_min10_pati5/run_summary.json
